# Q-Learning

Q-learning is a **model-free** RL algorithm that learns the optimal action-value function $Q^*(s, a)$ directly from experience, without needing a model of the environment.

The update rule is:

$$Q(s, a) \leftarrow Q(s, a) + \alpha \left[ r + \gamma \max_{a'} Q(s', a') - Q(s, a) \right]$$

where $\alpha$ is the learning rate and $\gamma$ is the discount factor.

In this notebook we:
1. Build a simple gridworld environment with numpy
2. Implement Q-learning from scratch
3. Visualize the Q-table, learning curves, and exploration behavior
4. Compare different hyperparameter settings

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style('whitegrid')
np.random.seed(42)

## 1. Gridworld Environment

A 5x5 grid with:
- Start at (0, 0)
- Goal at (4, 4) with reward +10
- A trap at (2, 2) with reward -10
- Step penalty of -0.1 to encourage shorter paths
- Walls at (1, 1) and (3, 3)

In [ ]:
class GridWorldQL:
    """A 5x5 gridworld for Q-learning."""
    
    ACTIONS = {0: (-1, 0), 1: (1, 0), 2: (0, -1), 3: (0, 1)}  # U, D, L, R
    ACTION_NAMES = ['Up', 'Down', 'Left', 'Right']
    
    def __init__(self, rows=5, cols=5):
        self.rows = rows
        self.cols = cols
        self.start = (0, 0)
        self.goal = (4, 4)
        self.trap = (2, 2)
        self.walls = {(1, 1), (3, 3)}
        self.state = self.start
    
    def reset(self):
        self.state = self.start
        return self.state
    
    def step(self, action):
        r, c = self.state
        dr, dc = self.ACTIONS[action]
        nr, nc = r + dr, c + dc
        
        # Boundary check
        if nr < 0 or nr >= self.rows or nc < 0 or nc >= self.cols:
            nr, nc = r, c
        
        # Wall check
        if (nr, nc) in self.walls:
            nr, nc = r, c
        
        self.state = (nr, nc)
        
        # Rewards
        if self.state == self.goal:
            return self.state, 10.0, True
        elif self.state == self.trap:
            return self.state, -10.0, True
        else:
            return self.state, -0.1, False


env = GridWorldQL()
print(f'Grid: {env.rows}x{env.cols}')
print(f'Start: {env.start}, Goal: {env.goal}, Trap: {env.trap}')
print(f'Walls: {env.walls}')

## 2. Q-Learning Implementation

In [ ]:
def q_learning(env, n_episodes=1000, alpha=0.1, gamma=0.95, 
               epsilon_start=1.0, epsilon_end=0.01, epsilon_decay=0.995,
               max_steps=100):
    """Train a Q-learning agent."""
    Q = np.zeros((env.rows, env.cols, 4))
    
    rewards_per_episode = []
    steps_per_episode = []
    epsilon_history = []
    epsilon = epsilon_start
    
    for episode in range(n_episodes):
        state = env.reset()
        total_reward = 0
        
        for step in range(max_steps):
            r, c = state
            
            # Epsilon-greedy action selection
            if np.random.rand() < epsilon:
                action = np.random.randint(4)
            else:
                action = np.argmax(Q[r, c])
            
            next_state, reward, done = env.step(action)
            nr, nc = next_state
            
            # Q-learning update
            best_next = np.max(Q[nr, nc])
            Q[r, c, action] += alpha * (reward + gamma * best_next * (1 - done) - Q[r, c, action])
            
            total_reward += reward
            state = next_state
            
            if done:
                break
        
        rewards_per_episode.append(total_reward)
        steps_per_episode.append(step + 1)
        epsilon_history.append(epsilon)
        epsilon = max(epsilon_end, epsilon * epsilon_decay)
    
    return Q, rewards_per_episode, steps_per_episode, epsilon_history


Q, rewards, steps, epsilons = q_learning(env, n_episodes=2000)
print(f'Training complete. Final epsilon: {epsilons[-1]:.4f}')
print(f'Average reward (last 100 episodes): {np.mean(rewards[-100:]):.2f}')

## 3. Learning Curve

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Reward per episode (smoothed)
window = 50
smoothed_rewards = np.convolve(rewards, np.ones(window)/window, mode='valid')
axes[0].plot(smoothed_rewards, color='#2ecc71', linewidth=1.5)
axes[0].set_xlabel('Episode')
axes[0].set_ylabel('Total Reward')
axes[0].set_title(f'Reward per Episode (smoothed, window={window})')
axes[0].axhline(y=10, color='red', linestyle='--', alpha=0.5, label='Max possible')
axes[0].legend()

# Steps per episode (smoothed)
smoothed_steps = np.convolve(steps, np.ones(window)/window, mode='valid')
axes[1].plot(smoothed_steps, color='#3498db', linewidth=1.5)
axes[1].set_xlabel('Episode')
axes[1].set_ylabel('Steps')
axes[1].set_title(f'Steps per Episode (smoothed, window={window})')

# Epsilon decay
axes[2].plot(epsilons, color='#e74c3c', linewidth=1.5)
axes[2].set_xlabel('Episode')
axes[2].set_ylabel('Epsilon')
axes[2].set_title('Exploration Rate (Epsilon) Over Time')

plt.tight_layout()
plt.show()

## 4. Q-Table Visualization

We visualize the maximum Q-value at each state (the value of the best action) and the learned policy.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Max Q-value heatmap
max_Q = np.max(Q, axis=2)
# Mark walls as NaN for display
display_Q = max_Q.copy()
for (wr, wc) in env.walls:
    display_Q[wr, wc] = np.nan

sns.heatmap(display_Q, annot=True, fmt='.2f', cmap='RdYlGn', 
            linewidths=2, linecolor='black', ax=axes[0],
            cbar_kws={'label': 'Max Q-Value'})
axes[0].set_title('Max Q-Value per State', fontsize=13)
axes[0].set_xlabel('Column')
axes[0].set_ylabel('Row')

# Q-values for each action as a grid of mini-heatmaps
action_names = ['Up', 'Down', 'Left', 'Right']
q_per_action = np.zeros((env.rows * 2, env.cols * 2))

for r in range(env.rows):
    for c in range(env.cols):
        # Arrange: Up (top), Down (bottom), Left (left), Right (right)
        q_per_action[r*2, c*2] = Q[r, c, 0]      # Up -> top-left
        q_per_action[r*2, c*2+1] = Q[r, c, 3]     # Right -> top-right
        q_per_action[r*2+1, c*2] = Q[r, c, 2]     # Left -> bottom-left
        q_per_action[r*2+1, c*2+1] = Q[r, c, 1]   # Down -> bottom-right

sns.heatmap(q_per_action, cmap='RdYlGn', linewidths=0.5, linecolor='gray',
            ax=axes[1], cbar_kws={'label': 'Q-Value'})
axes[1].set_title('Q-Values for All (state, action) Pairs', fontsize=13)
axes[1].set_xlabel('(state_col x action)')
axes[1].set_ylabel('(state_row x action)')

plt.tight_layout()
plt.show()

## 5. Learned Policy Visualization

In [ ]:
def plot_policy_ql(Q, env):
    """Plot the greedy policy derived from Q."""
    fig, ax = plt.subplots(figsize=(8, 8))
    
    arrow_symbols = ['\u2191', '\u2193', '\u2190', '\u2192']  # Up, Down, Left, Right
    arrow_dx = {0: 0, 1: 0, 2: -0.3, 3: 0.3}
    arrow_dy = {0: 0.3, 1: -0.3, 2: 0, 3: 0}
    
    # Draw grid
    for r in range(env.rows + 1):
        ax.axhline(y=r, color='black', linewidth=2)
    for c in range(env.cols + 1):
        ax.axvline(x=c, color='black', linewidth=2)
    
    for r in range(env.rows):
        for c in range(env.cols):
            x = c + 0.5
            y = (env.rows - 1 - r) + 0.5
            
            if (r, c) in env.walls:
                ax.add_patch(plt.Rectangle((c, env.rows - 1 - r), 1, 1,
                                            fill=True, color='black', alpha=0.7))
                ax.text(x, y, 'WALL', ha='center', va='center',
                        color='white', fontsize=9, fontweight='bold')
            elif (r, c) == env.goal:
                ax.add_patch(plt.Rectangle((c, env.rows - 1 - r), 1, 1,
                                            fill=True, color='green', alpha=0.3))
                ax.text(x, y, 'GOAL\n+10', ha='center', va='center',
                        color='green', fontsize=10, fontweight='bold')
            elif (r, c) == env.trap:
                ax.add_patch(plt.Rectangle((c, env.rows - 1 - r), 1, 1,
                                            fill=True, color='red', alpha=0.2))
                ax.text(x, y, 'TRAP\n-10', ha='center', va='center',
                        color='red', fontsize=10, fontweight='bold')
            else:
                a = np.argmax(Q[r, c])
                dx, dy = arrow_dx[a], arrow_dy[a]
                ax.annotate('', xy=(x + dx, y + dy), xytext=(x - dx, y - dy),
                            arrowprops=dict(arrowstyle='->', color='#2c3e50', lw=2.5))
    
    ax.set_xlim(0, env.cols)
    ax.set_ylim(0, env.rows)
    ax.set_aspect('equal')
    ax.set_title('Learned Policy (Q-Learning)', fontsize=14)
    ax.set_xticks(np.arange(env.cols) + 0.5)
    ax.set_xticklabels(range(env.cols))
    ax.set_yticks(np.arange(env.rows) + 0.5)
    ax.set_yticklabels(range(env.rows - 1, -1, -1))
    plt.tight_layout()
    plt.show()


plot_policy_ql(Q, env)

## 6. Hyperparameter Comparison

Let's compare how different learning rates and discount factors affect training.

In [ ]:
# Compare learning rates
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
window = 50

learning_rates = [0.01, 0.1, 0.5, 0.9]
colors = sns.color_palette('viridis', len(learning_rates))

for alpha, color in zip(learning_rates, colors):
    env_temp = GridWorldQL()
    _, rews, _, _ = q_learning(env_temp, n_episodes=2000, alpha=alpha, gamma=0.95)
    smoothed = np.convolve(rews, np.ones(window)/window, mode='valid')
    axes[0].plot(smoothed, label=f'alpha={alpha}', color=color, linewidth=1.5)

axes[0].set_xlabel('Episode')
axes[0].set_ylabel('Reward (smoothed)')
axes[0].set_title('Effect of Learning Rate (alpha)', fontsize=13)
axes[0].legend()

# Compare discount factors
discount_factors = [0.5, 0.8, 0.95, 0.99]
colors = sns.color_palette('magma', len(discount_factors))

for gamma, color in zip(discount_factors, colors):
    env_temp = GridWorldQL()
    _, rews, _, _ = q_learning(env_temp, n_episodes=2000, alpha=0.1, gamma=gamma)
    smoothed = np.convolve(rews, np.ones(window)/window, mode='valid')
    axes[1].plot(smoothed, label=f'gamma={gamma}', color=color, linewidth=1.5)

axes[1].set_xlabel('Episode')
axes[1].set_ylabel('Reward (smoothed)')
axes[1].set_title('Effect of Discount Factor (gamma)', fontsize=13)
axes[1].legend()

plt.tight_layout()
plt.show()

## 7. Exploration vs Exploitation

Visualize how the agent's behavior changes as epsilon decays.

In [ ]:
# Track visit counts during training
env_track = GridWorldQL()
Q_track = np.zeros((env_track.rows, env_track.cols, 4))
visit_maps = []  # snapshots of visit counts
visit_counts = np.zeros((env_track.rows, env_track.cols))

epsilon = 1.0
snapshot_episodes = [50, 200, 500, 1500]

for episode in range(2000):
    state = env_track.reset()
    for step in range(100):
        r, c = state
        visit_counts[r, c] += 1
        
        if np.random.rand() < epsilon:
            action = np.random.randint(4)
        else:
            action = np.argmax(Q_track[r, c])
        
        next_state, reward, done = env_track.step(action)
        nr, nc = next_state
        Q_track[r, c, action] += 0.1 * (reward + 0.95 * np.max(Q_track[nr, nc]) * (1 - done) - Q_track[r, c, action])
        state = next_state
        if done:
            break
    
    if (episode + 1) in snapshot_episodes:
        visit_maps.append((episode + 1, visit_counts.copy()))
    
    epsilon = max(0.01, epsilon * 0.995)

# Plot visit count heatmaps over time
fig, axes = plt.subplots(1, 4, figsize=(20, 5))

for ax, (ep, vmap) in zip(axes, visit_maps):
    sns.heatmap(vmap, annot=True, fmt='.0f', cmap='Blues',
                linewidths=1, linecolor='gray', ax=ax)
    ax.set_title(f'After {ep} Episodes', fontsize=12)
    ax.set_xlabel('Column')
    ax.set_ylabel('Row')

fig.suptitle('State Visit Counts: Exploration to Exploitation', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## Summary

- **Q-learning** is off-policy: it learns the optimal Q-function regardless of the exploration policy used.
- **Epsilon-greedy** exploration starts with high randomness and gradually shifts to exploitation.
- **Learning rate** (alpha): too low = slow convergence; too high = instability.
- **Discount factor** (gamma): higher values make the agent more far-sighted; lower values focus on immediate rewards.
- The visit count heatmaps show how exploration becomes more focused over time as the agent learns the optimal path.